# Задание 9: Тест на марковость 0-го vs 1-го порядка

Статистически проверяем зависимость между соседними нуклеотидами с помощью хи-квадрат теста.

In [1]:
import numpy as np
from scipy import stats
from Bio import SeqIO

## Загрузка реальной последовательности и создание случайной

In [2]:
nucleotides = ['A', 'C', 'G', 'T']
nuc_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

# Реальная последовательность
record = next(SeqIO.parse("GCA_029856635.1_ASM2985663v1_genomic.fna", "fasta"))
real_seq = ''.join(c for c in str(record.seq).upper() if c in 'ACGT')

print(f"Реальная последовательность: {len(real_seq)} нт")

# Случайная последовательность с теми же моно-нуклеотидными частотами 
mono_counts = np.array([real_seq.count(n) for n in nucleotides], dtype=float)
freqs = mono_counts / mono_counts.sum()

np.random.seed(42)
random_seq = ''.join(np.random.choice(nucleotides, size=len(real_seq), p=freqs))

print(f"Случайная последовательность: {len(random_seq)} нт")
print(f"Частоты: {dict(zip(nucleotides, freqs.round(4)))}")

Реальная последовательность: 37163 нт
Случайная последовательность: 37163 нт
Частоты: {'A': np.float64(0.3167), 'C': np.float64(0.1783), 'G': np.float64(0.1761), 'T': np.float64(0.3289)}


## Вспомогательные функции

In [3]:
def observed_counts(seq):
    """Матрица наблюдаемых счётов динуклеотидов O_ij."""
    O = np.zeros((4, 4), dtype=int)
    for i in range(len(seq) - 1):
        O[nuc_index[seq[i]], nuc_index[seq[i+1]]] += 1
    return O


def expected_counts(seq, O):
    """
    Ожидаемые частоты по модели 0-го порядка:
    E_ij = N * P(i) * P(j)
    """
    N = O.sum()
    mono = np.array([seq.count(n) for n in nucleotides], dtype=float)
    mono /= mono.sum()
    E = N * np.outer(mono, mono)
    return E


def chi2_test(O, E):
    """
    chi2 = sum((O - E)^2 / E),  df = 9
    """
    mask = E > 0
    chi2 = np.sum(((O[mask] - E[mask]) ** 2) / E[mask])
    pvalue = 1 - stats.chi2.cdf(chi2, df=9)
    return chi2, pvalue

## 1–4. Реальная последовательность

In [4]:
O_real = observed_counts(real_seq)
E_real = expected_counts(real_seq, O_real)
chi2_real, pval_real = chi2_test(O_real, E_real)

print("Наблюдаемые счёты O_ij (реальный геном):")
print("     ", "  ".join(nucleotides))
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  ", O_real[i])

print("\nОжидаемые счёты E_ij (модель 0-го порядка):")
print("     ", "       ".join(nucleotides))
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  ", [round(x, 1) for x in E_real[i]])

print(f"\nχ² = {chi2_real:.2f}")
print(f"p-value = {pval_real:.2e}")

Наблюдаемые счёты O_ij (реальный геном):
      A  C  G  T
A   [4382 1582 2109 3694]
C   [2134 1186 1003 2305]
G   [2306 1387 1188 1663]
T   [2946 2472 2244 4561]

Ожидаемые счёты E_ij (модель 0-го порядка):
      A       C       G       T
A   [np.float64(3726.3), np.float64(2098.8), np.float64(2072.2), np.float64(3870.4)]
C   [np.float64(2098.8), np.float64(1182.1), np.float64(1167.1), np.float64(2179.9)]
G   [np.float64(2072.2), np.float64(1167.1), np.float64(1152.3), np.float64(2152.3)]
T   [np.float64(3870.4), np.float64(2179.9), np.float64(2152.3), np.float64(4020.1)]

χ² = 798.94
p-value = 0.00e+00


## Случайная последовательность

In [5]:
O_rand = observed_counts(random_seq)
E_rand = expected_counts(random_seq, O_rand)
chi2_rand, pval_rand = chi2_test(O_rand, E_rand)

print("Наблюдаемые счёты O_ij (случайная последовательность):")
print("     ", "  ".join(nucleotides))
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  ", O_rand[i])

print(f"\nχ² = {chi2_rand:.2f}")
print(f"p-value = {pval_rand:.2e}")

Наблюдаемые счёты O_ij (случайная последовательность):
      A  C  G  T
A   [3787 2110 2142 3727]
C   [1972 1213 1192 2241]
G   [2090 1165 1150 2235]
T   [3917 2129 2156 3936]

χ² = 22.31
p-value = 7.96e-03


## 5. Итоговые выводы

In [6]:
ALPHA = 0.05

print("=" * 60)
print("ИТОГИ")
print("=" * 60)
print(f"{'Последовательность':28s} {'χ²':>10s} {'p-value':>12s} {'Вывод':>16s}")
print("-" * 68)
for name, chi2, pval in [("Реальный геном", chi2_real, pval_real),
                          ("Случайная (0-й порядок)", chi2_rand, pval_rand)]:
    verdict = "Отвергаем H0" if pval < ALPHA else "Не отвергаем H0"
    print(f"{name:28s} {chi2:>10.2f} {pval:>12.2e} {verdict:>16s}")

print()
print("H0: соседние нуклеотиды независимы (модель 0-го порядка достаточна).")
print()
if pval_real < ALPHA:
    print("Реальный геном: p-value < 0.05 — отвергаем H0.")
    print("Нуклеотиды НЕ независимы => необходима модель 1-го порядка.")
if pval_rand >= ALPHA:
    print("Случайная: p-value >= 0.05 — не отвергаем H0.")
    print("Нуклеотиды независимы (ожидаемо для модели 0-го порядка).")

ИТОГИ
Последовательность                   χ²      p-value            Вывод
--------------------------------------------------------------------
Реальный геном                   798.94     0.00e+00     Отвергаем H0
Случайная (0-й порядок)           22.31     7.96e-03     Отвергаем H0

H0: соседние нуклеотиды независимы (модель 0-го порядка достаточна).

Реальный геном: p-value < 0.05 — отвергаем H0.
Нуклеотиды НЕ независимы => необходима модель 1-го порядка.
